# 04. DeBERTa Fine-tuning

DeBERTa-v3-base with position bias mitigation and label smoothing.

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_cosine_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import log_loss
from pathlib import Path
import gc

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

train_df['target'] = (train_df['winner_model_a'].astype(int) * 0 +
                      train_df['winner_model_b'].astype(int) * 1 +
                      train_df['winner_tie'].astype(int) * 2)

In [ ]:
# Config
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LENGTH = 512
BATCH_SIZE = 8
GRAD_ACCUM = 4
EPOCHS = 3
LR = 2e-5
WARMUP_RATIO = 0.1
LABEL_SMOOTHING = 0.05

print(f'Device: {DEVICE}')
print(f'Model: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class ComparisonDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        resp_a = str(row['response_a'])
        resp_b = str(row['response_b'])
        target = row['target']

        # Position randomization during training
        if self.is_train and np.random.rand() < 0.5:
            resp_a, resp_b = resp_b, resp_a
            if target == 0:
                target = 1
            elif target == 1:
                target = 0

        encoding = self.tokenizer(
            prompt,
            resp_a + ' [SEP] ' + resp_b,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(target, dtype=torch.long)
        }

In [ ]:
# Split: last 10% for validation
val_size = int(0.1 * len(train_df))
train_subset = train_df.iloc[:-val_size]
val_df = train_df.iloc[-val_size:]

train_dataset = ComparisonDataset(train_subset, tokenizer, MAX_LENGTH, is_train=True)
val_dataset = ComparisonDataset(val_df, tokenizer, MAX_LENGTH, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler = GradScaler()
model.train()

print(f'Training: {len(train_dataset)} samples, {EPOCHS} epochs')

In [ ]:
for epoch in range(EPOCHS):
    epoch_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        epoch_loss += loss.item() * GRAD_ACCUM

    print(f'Epoch {epoch+1}: avg_loss={epoch_loss/len(train_loader):.4f}')

In [ ]:
# Validation predictions
model.eval()
val_preds = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        val_preds.append(probs)

val_preds = np.vstack(val_preds)
val_loss = log_loss(val_df['target'].values, val_preds)
print(f'Validation log_loss: {val_loss:.4f}')

In [ ]:
# Test predictions
test_dataset = ComparisonDataset(test_df, tokenizer, MAX_LENGTH, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

test_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        test_preds.append(probs)

test_preds = np.vstack(test_preds)
print(f'Test predictions shape: {test_preds.shape}')

In [ ]:
# Save OOF and test predictions (OOF is partial — only last 10%)
oof_preds = np.zeros((len(train_df), 3))
oof_preds[-val_size:] = val_preds

np.save(OUTPUT_DIR / 'deberta_oof.npy', oof_preds)
np.save(OUTPUT_DIR / 'deberta_test.npy', test_preds)

del model
gc.collect()
torch.cuda.empty_cache()

print('Saved DeBERTa predictions')